# 🔧 burst_scan — deep technical inspection

The other notebooks ask whether the strategy has an edge. This one asks whether **the machinery is right**, so that
their answers can be trusted — and what the book looks like from the inside.

| § | question |
|---|---|
| 1 | **label ↔ engine** — does vectorbt book every trade exactly the way the labels the model trained on describe it? |
| 2 | **look-ahead** — are events, features and predictions unchanged by later data? is the walk-forward purged? |
| 3 | **execution and costs** — what a trade really costs at the real order size; the book under 2× costs, pessimistic fills, late fills, whole shares and a borrow fee |
| 4 | **book anatomy** — how invested, how many positions, long vs short exposure, overnight vs intraday P&L, holding times, exit reasons, the engine's wallet rules |
| 5 | **concentration** — P&L by name, month, time of day and weekday |
| 6 | **runtime and memory** of one `signals()` call, stage by stage |
| 7 | the **holdout**, read once — off by default |

Everything before §7 uses the training window only. §7 stays off (`READ_HOLDOUT = False`) until a sub-strategy has
passed its gates and its cell is pinned; the holdout is then read **once** and never tuned on.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


import dataclasses
import resource
import time

SUBSET = None  # e.g. 40 for a quick first run; None = all 200 names
OVERRIDES = {}  # any signals() knob
PRM = {**strat.params, **OVERRIDES}
SCORE_FROM = strat.optuna["score_from"]
READ_HOLDOUT = False  # True ONLY after a side passed and its cell is pinned — read it once
HOLDOUT = ("2026-05-01", "2026-06-30")  # one symbol has later 1m bars; the holdout stops here
INIT_CASH = settings.backtest.init_cash

## 1 · Label ↔ engine reconciliation (Fig 1)

The classifier is trained on barrier labels computed by `strategy.outcomes()`; the P&L comes from vectorbt. They must
describe the **same trades**: every traded event joined to the engine's trade on (symbol, entry bar), with its exit
bar and exit price compared exactly (no costs, so prices are raw fills). The engine has **wallet rules** the labels
do not model, each named and priced below (`research.ENGINE_RULES`):

* an opposite-side signal on a held name **closes** that position instead of opening its own
  (`upon_opposite_entry="close"`) — `opposite close` / `closed an opposite position`;
* an entry on a bar where the shared wallet has **no free cash** left never fills — every slot is a share of the
  *free* cash, and open shorts tie cash up — `wallet had no free cash`;
* (since 2026-09-27 the engine LOCKS a short's sale proceeds for its cover, `lock_cash=True`; before that a long
  could spend them and a short's stop then covered only part of the position. A `cover limited by free cash` row
  would now be a defect);
* a label that runs past the last bar is `open at data end`.

Anything else that is not `ok` is a defect.

In [ ]:
panel = rs.load(settings, engine, subset=SUBSET)
c = compose(panel, **PRM)
ctx = c["ctx"]
rec = rs.reconcile(panel, c)
print(f"panel: {panel.close.shape[0]:,} {panel.bar} bars x {len(panel.symbols)} names")
display(rec.groupby(["side", "status"]).size().unstack(fill_value=0))
ok = rec[rec["status"] == "ok"]
gap = np.abs(ok["exit_price"] - ok["exit_px"]).max()
print(f"exact matches: max |exit price − label| = {gap:.2e}")
for rule in rs.ENGINE_RULES:
    hit = rec[rec["status"] == rule]
    if len(hit):
        avg = 1e4 * hit["ret"].mean()
        print(f"{rule:30s} {len(hit):5d} trades · their labels averaged {avg:+.1f} bp")
defects = rec[~rec["status"].isin(["ok", *rs.ENGINE_RULES])]
display(defects.head(10))

fig = go.Figure()
for side in SIDES:
    g = rec[rec["side"] == side]["status"].value_counts()
    fig.add_trace(go.Bar(x=g.index, y=g.to_numpy(), name=side, marker_color=COLOR[side]))
plots._base(fig, title="Fig 1 · every traded event against the engine's trade", height=360)
below(fig, -0.2)
fig.show()

## 2 · Look-ahead audit

**Truncation:** the whole pipeline re-run on a panel cut at a session boundary mid-window — every event, every
feature and every out-of-sample probability before the cut must be *identical* (a feature or a fold that peeks
ahead changes when the future is removed). **Delay:** the classifier re-run with every feature taken one bar
EARLIER than the event; stale features cannot beat fresh ones by much — a delay that *improves* the picks
would mean the event bar's own features carry something they should not. **Purge:** the flipped-label audit of
the model notebook, repeated here.

In [ ]:
codes = ctx["codes"]
cut_session = int(codes.max()) // 2
cut_row = int(np.flatnonzero(codes == cut_session)[0])
short_panel = dataclasses.replace(
    panel,
    close=panel.close.iloc[:cut_row],
    open_=panel.open_.iloc[:cut_row],
    high=panel.high.iloc[:cut_row],
    low=panel.low.iloc[:cut_row],
    volume=panel.volume.iloc[:cut_row],
)
c_cut = compose(short_panel, **PRM)
look = {}
for side in SIDES:
    a = c["events"][side]
    a = a[a["row"] < cut_row].reset_index(drop=True)
    b = c_cut["events"][side]
    same_ev = (
        len(a) == len(b) and (a[["row", "col"]].to_numpy() == b[["row", "col"]].to_numpy()).all()
    )
    same_x = same_ev and np.allclose(
        a[list(S["FEATURES"])].to_numpy(float),
        b[list(S["FEATURES"])].to_numpy(float),
        equal_nan=True,
    )
    same_p = same_ev and np.allclose(a["p"].to_numpy(float), b["p"].to_numpy(float), equal_nan=True)
    t = c["events"][side]
    rows, cols = t["row"].to_numpy(), t["col"].to_numpy()
    X_lag = S["features"](ctx, np.maximum(rows - 1, 0), cols).to_numpy(float)
    lag = rs.refit(t, ctx, PRM, side, X=X_lag)
    pk_now = t[t["picked"] & t["p"].notna()]
    pk_lag = t[lag["picked"].to_numpy() & lag["p"].notna().to_numpy()]
    look[side] = {
        "events before the cut": len(a),
        "events identical": bool(same_ev),
        "features identical": bool(same_x),
        "p identical": bool(same_p),
        "picks bp (features at t)": 1e4 * pk_now["dret"].mean(),
        "picks bp (features at t−1)": 1e4 * pk_lag["dret"].mean(),
    }
display(pd.DataFrame(look))
purge = pd.concat([rs.purge_audit(c["events"][s], ctx, PRM, folds=3).assign(side=s) for s in SIDES])
purge

## 3 · Execution and costs

**Per trade:** the engine charges ONE fee fraction per book, priced at the panel's median close and at the
strategy's `order_notional`; a real order pays `max($1, shares × $0.005)` + half the spread + slippage per leg, so
cheap names pay more commission per dollar and expensive ones hit the $1 minimum. **Stresses:** the same book with
every cost doubled, with pessimistic fills (buys at the bar's high, sells at its low), with every fill 50 bp late,
in whole shares, and with a 3%/yr borrow fee on every night a short is held (the engine models none).

In [ ]:
fees, slip = bt.strategy_costs(strat, panel, settings)
pf = bt.run_strategy(strat, panel, PRM, settings=settings, score_from=SCORE_FROM)
trades = bt.trade_records(pf)
anat = rs.cost_anatomy(
    trades, settings.costs, notional=strat.order_notional, fees=fees, slippage=slip
)
display(anat[["commission_bp", "real_rt_bp", "engine_rt_bp"]].describe().round(2).T)
real_rt, engine_rt = anat["real_rt_bp"].median(), anat["engine_rt_bp"].iloc[0]
print(
    f"minimum commission binds on {anat['min_binds'].mean():.0%} of trades; "
    f"real round trip {real_rt:.1f} bp median vs the engine's {engine_rt:.1f} bp"
)


def costs_x(mult=1.0, extra_slip_bps=0.0):
    k = settings.costs
    new = k.model_copy(
        update={
            "commission_per_share": k.commission_per_share * mult,
            "min_commission": k.min_commission * mult,
            "spread_bps": k.spread_bps * mult,
            "slippage_bps": k.slippage_bps * mult + extra_slip_bps,
        }
    )
    return settings.model_copy(update={"costs": new})


def stress(label, strategy=strat, sett=settings):
    st = bt.evaluate(strategy, panel, PRM, settings=sett, score_from=SCORE_FROM)
    return {
        "case": label,
        **{m: st[m] for m in ("cagr", "alpha_cagr", "beta", "excess_cagr", "n_trades")},
    }


cases = [
    stress("as configured"),
    stress("2x every cost", sett=costs_x(2.0)),
    stress("pessimistic fills", strategy=dataclasses.replace(strat, fill="pessimistic")),
    stress("fills 50 bp late", sett=costs_x(1.0, 50.0)),
    stress("whole shares", strategy=dataclasses.replace(strat, whole_shares=True)),
]
shorts = trades[trades["direction"] == "Short"]
nights = (
    shorts["exit_ts"].dt.tz_convert(panel.tz).dt.normalize()
    - shorts["entry_ts"].dt.tz_convert(panel.tz).dt.normalize()
).dt.days
borrow = float((shorts["size"].abs() * shorts["entry_price"] * nights * 0.03 / 365).sum())
years = bt._years(pf.wrapper.index)
cases.append(
    {
        **cases[0],
        "case": "3%/yr borrow on short nights",
        "cagr": cases[0]["cagr"] - borrow / INIT_CASH / years,
        "alpha_cagr": cases[0]["alpha_cagr"] - borrow / INIT_CASH / years,
    }
)
pd.DataFrame(cases).set_index("case").round(4)

## 4 · Book anatomy (Fig 4)

**Top:** gross exposure (long + short, as a share of the book) and the number of open positions. **Middle:** gross long and short
exposure. **Bottom:** the traded events' return split into the overnight gaps they held and the rest (bp per
trade, per side) — these names made their money overnight, so a long book that earns only overnight is running
exposure, not detection. The tables: holding times, exit reasons (from the labels), and the size of the orders
(the geometric `cash_frac` sizing can leave dust positions when the wallet is nearly full).

In [ ]:
value = pf.value()
asset = pf.asset_value(group_by=False)
long_x = asset.clip(lower=0).sum(axis=1) / value
short_x = -asset.clip(upper=0).sum(axis=1) / value
invested = long_x + short_x  # GROSS: 1 − cash/value would net short proceeds against longs
held = pd.Series(0, index=value.index, dtype=float)
ix = value.index
ent = ix.get_indexer(trades["entry_ts"])
ext = ix.get_indexer(trades["exit_ts"].where(trades["status"] == "Closed", ix[-1]))
delta = np.zeros(len(ix) + 1)
np.add.at(delta, ent, 1)
np.add.at(delta, ext, -1)
held[:] = np.cumsum(delta[:-1])
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("gross exposure and open positions (right)", "gross exposure: long and short"),
    specs=[[{"secondary_y": True}], [{}]],
)
fig.add_trace(
    plots._scatter(invested, name="gross exposure", line={"color": plots.SERIES[4], "width": 1}),
    row=1,
    col=1,
)
fig.add_trace(
    plots._scatter(held, name="open positions", line={"color": plots._MUTED, "width": 1}),
    row=1,
    col=1,
    secondary_y=True,
)
fig.add_trace(
    plots._scatter(long_x, name="long", line={"color": plots._LONG, "width": 1}), row=2, col=1
)
fig.add_trace(
    plots._scatter(short_x, name="short", line={"color": plots._SHORT, "width": 1}), row=2, col=1
)
plots._base(fig, title="Fig 4 · the book from the inside", height=560)
below(fig, -0.08)
fig.update_xaxes(type="date")
fig.update_yaxes(tickformat=".0%", row=1, col=1, secondary_y=False)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.show()

inside = {}
for side in SIDES:
    t = c["events"][side]
    t = t[t["traded"] & t["complete"]]
    parts = rs.split_overnight(panel, ctx, t, side)
    inside[side] = {
        "trades": len(t),
        "total bp": 1e4 * parts["total"].mean(),
        "overnight bp": 1e4 * parts["overnight"].mean(),
        "intraday bp": 1e4 * parts["intraday"].mean(),
        "held overnight": t["nights"].gt(0).mean(),
        "median hold (bars)": t["bars"].median(),
        "target": t["reason"].eq(1).mean(),
        "stop": t["reason"].eq(-1).mean(),
        "time": t["reason"].eq(0).mean(),
    }
display(pd.DataFrame(inside).round(3))
order = trades["size"].abs() * trades["entry_price"]
print(
    f"orders: median ${order.median():,.0f}, {(order < 100).mean():.1%} below $100 (dust); "
    f"same-bar clashes dropped: {c['clash']}; average gross exposure {invested.mean():.1%}"
)
st = bt.evaluate(strat, panel, PRM, settings=settings, score_from=SCORE_FROM)
pd.Series(
    {
        m: st[m]
        for m in (
            "cagr",
            "alpha_cagr",
            "beta",
            "excess_cagr",
            "volatility",
            "sharpe",
            "max_drawdown",
            "trades_per_year",
        )
    }
).round(4)

## 5 · Concentration (Fig 5)

The book's closed-trade P&L by name (the top names, and the total without the five best), by month, by the hour
the trade was entered and by weekday. An edge that lives in five names or one month is an anecdote.

In [ ]:
closed = trades[trades["status"] == "Closed"].copy()
closed["month"] = closed["entry_ts"].dt.tz_convert(panel.tz).dt.strftime("%Y-%m")
closed["hour"] = closed["entry_ts"].dt.tz_convert(panel.tz).dt.hour
closed["weekday"] = closed["entry_ts"].dt.tz_convert(panel.tz).dt.day_name().str[:3]
by_name = closed.groupby("symbol")["pnl"].sum().sort_values()
top5 = by_name.nlargest(5)
print(
    f"closed-trade P&L ${closed['pnl'].sum():,.0f}; the five best names ${top5.sum():,.0f} "
    f"({', '.join(top5.index)}); without them ${closed['pnl'].sum() - top5.sum():,.0f}"
)
fig = make_subplots(
    rows=1, cols=3, subplot_titles=("by month ($)", "by entry hour ($)", "by weekday ($)")
)
for i, (key, order_) in enumerate(
    (("month", None), ("hour", None), ("weekday", ["Mon", "Tue", "Wed", "Thu", "Fri"])), start=1
):
    g = closed.groupby([key, "direction"])["pnl"].sum().unstack(fill_value=0)
    if order_:
        g = g.reindex(order_).fillna(0)
    for d, colr in (("Long", plots._LONG), ("Short", plots._SHORT)):
        if d in g:
            fig.add_trace(
                go.Bar(
                    x=g.index.astype(str),
                    y=g[d],
                    name=d.lower(),
                    marker_color=colr,
                    showlegend=i == 1,
                ),
                row=1,
                col=i,
            )
fig.update_layout(barmode="relative")
plots._base(fig, title="Fig 5 · where the P&L comes from", height=380)
below(fig, -0.2)
fig.show()
pd.concat([by_name.head(5), by_name.tail(5)]).rename("pnl $").round(0)

## 6 · Runtime and memory of one `signals()` call

Stage by stage on this panel, so the cost of a phase-2 search is known before it starts (a trial runs all of it).

In [ ]:
timing = {}
t0 = time.perf_counter()
ctx_t = S["context"](panel, **PRM)
timing["context (profiles, features)"] = time.perf_counter() - t0
for side in SIDES:
    t0 = time.perf_counter()
    r, c_ = rs.events(ctx_t, PRM, side)
    timing[f"{side}: detect"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    X = S["features"](ctx_t, r, c_)
    q = S["_side_params"](PRM, side)
    frac = q["move_atr"] * X["atr_pct"].to_numpy(float) * np.sqrt(q["horizon"] / 390)
    oc = S["outcomes"](
        panel,
        ctx_t,
        r,
        c_,
        side=1 if side == "long" else -1,
        horizon=S["_bars"](q["horizon"], panel.bar),
        frac=frac,
        eod=q["eod_exit"],
    )
    timing[f"{side}: features + outcomes"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    S["walk_forward"](
        X.to_numpy(float),
        oc["win"].to_numpy(),
        r,
        oc["exit_row"].to_numpy(),
        oc["complete"].to_numpy(),
        ctx_t,
        **rs.wf_kwargs(PRM, side),
    )
    timing[f"{side}: walk-forward ({PRM['model']})"] = time.perf_counter() - t0
t0 = time.perf_counter()
strat.signals(panel, **PRM)
timing["signals() end to end"] = time.perf_counter() - t0
del ctx_t
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9  # bytes on macOS
print(f"peak resident memory of this kernel so far: {peak:.1f} GB")
pd.Series(timing, name="seconds").round(2)

## 7 · The holdout (read once)

Off by default. When a sub-strategy has passed G0 and gates a–f and its cell is pinned, set `READ_HOLDOUT = True`,
run this cell once, and record the numbers in the README. The panel is reloaded through the holdout end (the
walk-forward keeps training on everything before each month, as it would live) and scored from the holdout start.

In [ ]:
if READ_HOLDOUT:
    full = rs.load(settings, engine, subset=SUBSET, end=HOLDOUT[1])
    hold = {}
    for sides in ("long", "short", "both"):
        st = bt.evaluate(
            strat, full, {**PRM, "sides": sides}, settings=settings, score_from=HOLDOUT[0]
        )
        hold[sides] = {
            m: st[m] for m in ("cagr", "alpha_cagr", "beta", "excess_cagr", "n_trades", "years")
        }
    display(pd.DataFrame(hold).round(4))
else:
    hold = None
    print("holdout NOT read (READ_HOLDOUT = False)")

## Checks

In [ ]:
explained = defects.empty
lk = pd.DataFrame(look)
rows = [
    (
        "engine trades = labels (up to the engine's wallet rules)",
        f"{len(defects)} defects",
        0,
        explained,
    ),
    (
        "events unchanged by later data",
        bool(lk.loc["events identical"].all()),
        True,
        bool(lk.loc["events identical"].all()),
    ),
    (
        "features unchanged by later data",
        bool(lk.loc["features identical"].all()),
        True,
        bool(lk.loc["features identical"].all()),
    ),
    (
        "predictions unchanged by later data",
        bool(lk.loc["p identical"].all()),
        True,
        bool(lk.loc["p identical"].all()),
    ),
    (
        "purge: flipped running labels change nothing",
        bool(purge["unchanged"].all()),
        True,
        bool(purge["unchanged"].all()),
    ),
    ("holdout untouched unless READ_HOLDOUT", READ_HOLDOUT, "False until pinned", True),
]
display(rs.checks_table(rows))
rs.scorecard(
    {
        "notebook": "diagnostics",
        "study": strat.optuna["study"],
        "names": len(panel.symbols),
        "reconcile": rec["status"].value_counts().to_dict(),
        "stress": {r["case"]: {"alpha": r["alpha_cagr"], "cagr": r["cagr"]} for r in cases},
        "anatomy": inside,
        "holdout": hold,
    }
)